In [0]:
container_path = r'abfss://sales-store@adlssalesstoree.dfs.core.windows.net/'
file_path_compras_presencial = container_path + 'landing/compras/Presencial.csv'
file_path_compras_online = container_path + 'landing/compras/Online.json'
folder_path_detalles = container_path + 'landing/detalles'


In [0]:
from pyspark.sql.functions import col, lit, current_timestamp, when

def cargar_compras_presencial(file_path):
    sales_pre = (
        spark.read.format("csv")
        .option("header", True)
        .option("sep", ";")
        .option("inferSchema", "false")
        .load(file_path)
    )

    df_presencial = (
        sales_pre.select(
            col("VentaID").alias("venta_id"),
            col("Factura").alias("factura"),
            col("Fecha_Orden").alias("fecha_orden"),
            col("Fecha_Entrega").alias("fecha_entrega"),
            col("Fecha_Envio").alias("fecha_envio"),
            col("Estado").alias("estado"),
            col("Cliente_code").alias("cliente_code"),
            col("Tipo_Cliente").alias("tipo_cliente"),
            col("Nombres").alias("nombres"),
            col("Apellidos").alias("apellidos"),
            col("Vendedor").alias("vendedor"),
            col("Departamento").alias("departamento"),
            col("Metodo_Pago").alias("metodo_pago"),
            col("Subtotal").alias("subtotal"),
            lit("Presencial").alias("tipo_compra"),
            current_timestamp().alias("fecha_carga")
        )
    )

    return df_presencial

def cargar_compras_online(file_path):
    sales_online = (
        spark.read.format("json")
        .option("multiline", "true")
        .load(file_path)
    )

    df_compras_online = (
        sales_online.select(
            col("VentaID").alias("venta_id"),
            col("Factura").alias("factura"),
            col("FechaOrden").alias("fecha_orden"),
            col("FechaEntrega").alias("fecha_entrega"),
            col("FechaEnvio").alias("fecha_envio"),
            col("Estado").alias("estado"),
            col("ClienteCode").alias("cliente_code"),
            col("TipoCliente").alias("tipo_cliente"),
            col("Nombres").alias("nombres"),
            col("Apellidos").alias("apellidos"),
            col("Vendedor").alias("vendedor"),
            col("Departamento").alias("departamento"),
            col("MetodoPago").alias("metodo_pago"),
            lit("Online").alias("tipo_compra"),
            current_timestamp().alias("fecha_carga")
        )
    )

    return df_compras_online

df_compras_presencial_df = cargar_compras_presencial(file_path_compras_presencial)
df_compras_online_df = cargar_compras_online(file_path_compras_online)
df_compras = df_compras_presencial_df.unionByName(df_compras_online_df, allowMissingColumns=True)
df_compras.write.format("delta").mode("overwrite").saveAsTable("azdb_sales_store.linio.bronze_compras")

def cargar_compras_detalle(file_path):
    sales_detalles = (
        spark.read.format("csv")
        .option("header", True)
        .option("sep", "|")
        .option("inferSchema", "false")
        .load(file_path)
    )

    df_dtll = (
        sales_detalles.select(
            col("Detalle_ID").alias("detalle_id"),
            col("Factura").alias("factura"),
            col("Num_Tracking").alias("num_tracking"),
            col("categoria").alias("categoria"),
            col("Subcategoria").alias("subcategoria"),
            col("Producto_ID").alias("producto_id"),
            col("Producto").alias("producto"),
            col("Unidades").alias("unidades"),
            col("Precio_Unitario").alias("precio_unitario"),
            col("Oferta_ID").alias("oferta_id"),
            col("_metadata.file_name").alias("nombre_archivo"),
            current_timestamp().alias("fecha_carga")
        )
    )

    return df_dtll


df_detalles = cargar_compras_detalle(folder_path_detalles)
df_detalles.write.format("delta").mode("overwrite").saveAsTable("azdb_sales_store.linio.bronze_detalles")


